In [ ]:
# ===============================
# 🦴 Bone TB Detection using CNN
# Google Colab Hackathon Starter
# ===============================

!pip install tensorflow opencv-python matplotlib

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import zipfile, os

In [ ]:
from google.colab import files
files.upload()  # upload the kaggle.json file you just downloaded

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d tawsifurrahman/tuberculosis-tb-chest-xray-dataset -p /content/data

In [ ]:
!unzip -q /content/data/tuberculosis-tb-chest-xray-dataset.zip -d /content/data/tb_cleaned

In [ ]:
!ls /content/data/tb_cleaned

In [ ]:
train_dir = "/content/data/tb_cleaned/TB_Chest_Radiography_Database/"
val_dir = "/content/data/tb_cleaned/TB_Chest_Radiography_Database/"

print("Training directory:", train_dir)
print("Validation directory:", val_dir)

In [ ]:
!ls /content/data/tb_cleaned/TB_Chest_Radiography_Database/Normal | head
!ls /content/data/tb_cleaned/TB_Chest_Radiography_Database/Tuberculosis | head

In [ ]:
# Data augmentation for better generalization
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)


In [ ]:
!ls -R /content/data/tb_cleaned/TB_Chest_Radiography_Database/

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')  # Binary classification
])

model.summary()


In [ ]:
from tensorflow.keras import optimizers

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    verbose=1
)


In [ ]:
loss, acc = model.evaluate(val_data)
print(f"\n✅ Validation Accuracy: {acc*100:.2f}%")


In [ ]:
model.save("bone_tb_cnn_model.h5")
print("💾 Model saved as bone_tb_cnn_model.h5")


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_image(img_path):
    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    prediction = model.predict(img_array)[0][0]
    label = "TB Positive" if prediction > 0.5 else "TB Negative"
    print(f"🩻 Prediction: {label} (Confidence: {prediction:.2f})")
    plt.imshow(img)
    plt.title(label)
    plt.axis('off')
    plt.show()


In [ ]:
from google.colab import files

print("📤 Upload an image to test prediction:")
uploaded_img = files.upload()

for img_name in uploaded_img.keys():
    predict_image(img_name)

In [ ]:
# ===========================
# 🚀 Transfer Learning: ResNet50
# ===========================

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

# Load pre-trained ResNet50 model (exclude top layers)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Freeze base model layers
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)  # Binary classification

# Final model
resnet_model = Model(inputs=base_model.input, outputs=predictions)
resnet_model.summary()

# Compile model
resnet_model.compile(optimizer=Adam(learning_rate=0.0001),
                     loss='binary_crossentropy',
                     metrics=['accuracy'])

In [ ]:
# ===========================
# 🌈 Grad-CAM Visualization
# ===========================

import numpy as np
import matplotlib.cm as cm
from tensorflow.keras.preprocessing import image
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
from google.colab import files


def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # Get model gradient model (inputs → conv layer → predictions)
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    # Compute gradients of predicted class w.r.t. conv layer outputs
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)

    # Average gradients over all pixels
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    # Weight output feature maps by importance
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def display_gradcam(img_path, model, last_conv_layer_name="conv5_block3_out"):
    # Load and preprocess image
    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    # Make Grad-CAM heatmap
    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)

    # Display heatmap superimposed on original image
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224,224))
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)

    heatmap_colored = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    superimposed_img = cv2.addWeighted(img, 0.6, heatmap_colored, 0.4, 0)

    # Convert BGR → RGB for plotting
    plt.figure(figsize=(6,6))
    plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
    plt.title("Grad-CAM Heatmap")
    plt.axis("off")
    plt.show()

# === Run Demo ===
print("📤 Upload an image to visualize Grad-CAM heatmap:")
uploaded_cam = files.upload()

for img_name in uploaded_cam.keys():
    display_gradcam(img_name, resnet_model, last_conv_layer_name="conv5_block3_out")  # use last conv layer name

In [ ]:
# ===========================
# 🚀 Transfer Learning: ResNet50
# ===========================

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout

# Load pre-trained ResNet50 model (exclude top layers)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Freeze base model layers
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)  # Binary classification

# Final model
resnet_model = Model(inputs=base_model.input, outputs=predictions)
resnet_model.summary()

# Compile model
resnet_model.compile(optimizer=Adam(learning_rate=0.0001),
                     loss='binary_crossentropy',
                     metrics=['accuracy'])


In [ ]:
# Train the ResNet50 model
history_resnet = resnet_model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,   # fewer epochs for demo
    verbose=1
)


In [ ]:
loss, acc = resnet_model.evaluate(val_data)
print(f"\n✅ Validation Accuracy (ResNet50): {acc*100:.2f}%")


In [ ]:
resnet_model.save("bone_tb_resnet50_model.h5")
print("💾 ResNet50 model saved as bone_tb_resnet50_model.h5")
